In [10]:
from google.colab import drive
drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/ImageDataset"

Mounted at /content/drive


In [11]:
import tensorflow as tf
img_height, img_width = 224, 224
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_path,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  dataset_path,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

class_names = train_ds.class_names

Found 5455 files belonging to 20 classes.
Using 4364 files for training.
Found 5455 files belonging to 20 classes.
Using 1091 files for validation.


In [13]:
from tensorflow.keras import layers, models

def build_model(base_model):
  base_model.trainable = False # Transfer Learning

  model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names))
])

  return model

In [14]:
from tensorflow.keras.applications import MobileNetV2, EfficientNetB3, NASNetMobile

models_dict = {
    "MobileNetV2": MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    "EfficientNetB3": EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    "NASNetMobile": NASNetMobile(weights='imagenet', include_top=False, input_shape=(224,224,3))
}

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
19993432/19993432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
from tensorflow.keras.optimizers import Adam

histories = {}
trained_models = {}

for name, base in models_dict.items():
    print(f"Training {name}...")
    model = build_model(base)
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    history = model.fit(        # ← make sure this is indented
        train_ds,               # ← inside the for loop
        validation_data=val_ds,
        epochs=13
    )
    histories[name] = history   # ← same here
    trained_models[name] = model # ← and here

Training MobileNetV2...
Epoch 1/13


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


137/137 ━━━━━━━━━━━━━━━━━━━━ 1150s 8s/step - accuracy: 0.1622 - loss: 2.8266 - val_accuracy: 0.4280 - val_loss: 2.2916
Epoch 2/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 373s 3s/step - accuracy: 0.3811 - loss: 2.1288 - val_accuracy: 0.6269 - val_loss: 1.6508
Epoch 3/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 344s 3s/step - accuracy: 0.5151 - loss: 1.6751 - val_accuracy: 0.7214 - val_loss: 1.2699
Epoch 4/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 393s 3s/step - accuracy: 0.6137 - loss: 1.3720 - val_accuracy: 0.7709 - val_loss: 1.0347
Epoch 5/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - accuracy: 0.6703 - loss: 1.1477 - val_accuracy: 0.7883 - val_loss: 0.8811
Epoch 6/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 345s 2s/step - accuracy: 0.7202 - loss: 1.0170 - val_accuracy: 0.8057 - val_loss: 0.7883
Epoch 7/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 357s 3s/step - accuracy: 0.7413 - loss: 0.9246 - val_accuracy: 0.8139 - val_loss: 0.7251
Epoch 8/13
137/137 ━━━━━━━━━━━━━━━━━━━━ 374s 3s/step - accuracy: 0.7599 - loss: 0.8582 - val_accuracy: 0.83

In [16]:
from sklearn.metrics import classification_report, confusion_matrix
def evaluate_model(model):

  y_true, y_pred = [], []
  for images, labels in val_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.argmax(axis=1))
  print(classification_report(y_true, y_pred))
  return y_true, y_pred

In [17]:
import matplotlib.pyplot as plt
import numpy as np
def plot_confusion_matrix(y_true, y_pred, title):
  cm = confusion_matrix(y_true, y_pred)
  plt.imshow(cm)
  plt.title(title)
  plt.colorbar()
  plt.xticks(np.arange(len(class_names)), class_names, rotation=45)
  plt.yticks(np.arange(len(class_names)), class_names)
  for i in range(len(class_names)):
     for j in range(len(class_names)):
       plt.text(j, i, cm[i, j], ha='center')
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.show()

In [18]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
def plot_roc(model):
  y_true, y_prob = [], []
  for images, labels in val_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_prob.extend(preds)

  y_true = np.array(y_true)
  y_prob = np.array(y_prob)

  y_bin = label_binarize(y_true, classes=range(len(class_names)))
  for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr)
  plt.title("ROC Curve")
  plt.show()

In [19]:
import tensorflow as tf
import numpy as np

def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [20]:
for name, model in trained_models.items():
    model.save(f"/content/drive/MyDrive/{name}_model.keras")
    print(f"✅ Saved: {name}")

✅ Saved: MobileNetV2
✅ Saved: EfficientNetB3
✅ Saved: NASNetMobile
